In [1]:
import numpy as np
import pandas as pd
from itertools import product

def concentration_score(file_R0='R0.csv', 
                        file_sigma='sigma.csv',
                        true_R0=None,
                        true_sigma=None,
                        selected_indices=None):

    data_R0 = pd.read_csv(file_R0, header=None).values.ravel()
    data_sigma = pd.read_csv(file_sigma, header=None).values.ravel()
    
    # print(f"Loaded {len(data_R0)} {len(data_sigma)} samples")
    selected_R0 = data_R0[selected_indices]
    selected_sigma = data_sigma[selected_indices]
    
    d_R0    = ((selected_R0 - true_R0)    ** 2).mean()
    d_sigma = ((selected_sigma - true_sigma) ** 2).mean()
    return d_R0 + d_sigma

def compute_distance(filepath: str,
    standard_point: tuple,
    weights: tuple = None,
    metric: str = "euclidean",  # "euclidean", "manhattan", "chebyshev", "minkowski", "cosine"
    p: float = 3,               # only used when metric="minkowski"
    ):
                     
     # 1. Load
    df = pd.read_csv(filepath, header=None, names=["A", "B", "C", "D"])

    # 2. Min-Max normalization
    col_min = df.min()
    col_max = df.max()
    df_norm = (df - col_min) / (col_max - col_min)

    # 3. Normalize the standard point on the same scale
    standard = np.array(standard_point)
    standard_norm = (standard - col_min.values) / (col_max.values - col_min.values)

    # 4. Resolve weights (normalize so they sum to 1)
    if weights is not None:
        w = np.array(weights, dtype=float)
        if len(w) != 4:
            raise ValueError("weights must have exactly 4 values (wA, wB, wC, wD).")
        if np.any(w < 0):
            raise ValueError("All weights must be non-negative.")
        w = w / w.sum()          # normalize to sum = 1
    else:
        w = np.array([0.25, 0.25, 0.25, 0.25])   # equal weights

    # 5. Weighted different distance functions: Euclidean, manhattan, chebyshev, minkowski, cosine
    diff = (df_norm[["A", "B", "C", "D"]] - standard_norm).values
    if metric == "euclidean":
        dist = np.sqrt((w * diff ** 2).sum(axis=1))

    elif metric == "manhattan":
        dist = (w * np.abs(diff)).sum(axis=1)

    elif metric == "chebyshev":
        dist = (w * np.abs(diff)).max(axis=1)

    elif metric == "minkowski":
        dist = ((w * np.abs(diff) ** p).sum(axis=1)) ** (1 / p)

    elif metric == "cosine":
        dot     = (w * df_norm[["A", "B", "C", "D"]].values * standard_norm).sum(axis=1)
        norm_x  = np.sqrt((w * df_norm[["A", "B", "C", "D"]].values ** 2).sum(axis=1))
        norm_x0 = np.sqrt((w * standard_norm ** 2).sum())
        dist    = 1 - dot / (norm_x * norm_x0)

    else:
        raise ValueError(f"Unknown metric '{metric}'. Choose from: euclidean, manhattan, chebyshev, minkowski, cosine.")

    df_norm["distance"] = dist

    return df_norm["distance"]



In [3]:
stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 5.0, 0.5   # ← your true values
percentile = 0.05

# Grid over weights
weight_values = [i/20 for i in range(1, 20)]  # 0.05, 0.10, ..., 0.95
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath="../../experimental_data/from_260312/ss_2params_R05p0.csv",
                                   standard_point=(192.95652174, 96.77926067, -0.36994225,  15.68656357),
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0='../../experimental_data/from_260312/R0_samps_2params_R05p0.csv',
                                file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R05p0.csv',
                                true_R0=5.0,
                                true_sigma=0.5, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                          weights     score     n
116637     (0.9, 0.05, 0.1, 0.8)  0.486904  5000
123497   (0.95, 0.05, 0.1, 0.85)  0.487133  5000
123496    (0.95, 0.05, 0.1, 0.8)  0.487448  5000
116638    (0.9, 0.05, 0.1, 0.85)  0.487451  5000
61739    (0.5, 0.05, 0.05, 0.45)  0.487676  5000
102919     (0.8, 0.05, 0.1, 0.8)  0.487833  5000
116635     (0.9, 0.05, 0.1, 0.7)  0.488265  5000
96057    (0.75, 0.05, 0.1, 0.65)  0.488319  5000
109776    (0.85, 0.05, 0.1, 0.7)  0.488338  5000
123498    (0.95, 0.05, 0.1, 0.9)  0.488357  5000
102917     (0.8, 0.05, 0.1, 0.7)  0.488451  5000
109779   (0.85, 0.05, 0.1, 0.85)  0.488776  5000
124220    (0.95, 0.15, 0.1, 0.9)  0.488835  5000
102916    (0.8, 0.05, 0.1, 0.65)  0.488988  5000
117359     (0.9, 0.15, 0.1, 0.8)  0.489274  5000
109777   (0.85, 0.05, 0.1, 0.75)  0.489531  5000
110137     (0.85, 0.1, 0.1, 0.7)  0.489754  5000
62100     (0.5, 0.1, 0.05, 0.45)  0.490037  5000
92090     (0.7, 0.45, 0.1, 0.85)  0.490136  500

In [7]:
stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 5.0, 0.5   # ← your true values
percentile = 0.05

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath="../../experimental_data/from_260312/ss_2params_R05p0.csv",
                                   standard_point=(192.95652174, 96.77926067, -0.36994225,  15.68656357),
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0='../../experimental_data/from_260312/R0_samps_2params_R05p0.csv',
                                file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R05p0.csv',
                                true_R0=5.0,
                                true_sigma=0.5, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

manhattan distance: 
                    weights     score     n
3658  (0.6, 0.1, 0.2, 0.5)  0.496594  5000
3010  (0.5, 0.2, 0.2, 0.5)  0.502727  5000
5127  (0.8, 0.1, 0.3, 0.7)  0.502785  5000
5856  (0.9, 0.1, 0.3, 0.7)  0.503678  5000
5858  (0.9, 0.1, 0.3, 0.9)  0.505661  5000
5938  (0.9, 0.2, 0.3, 0.8)  0.507126  5000
4560  (0.7, 0.3, 0.3, 0.7)  0.507136  5000
3912  (0.6, 0.4, 0.3, 0.7)  0.507793  5000
5209  (0.8, 0.2, 0.3, 0.8)  0.508300  5000
5208  (0.8, 0.2, 0.3, 0.7)  0.508648  5000
6182  (0.9, 0.5, 0.3, 0.9)  0.508882  5000
2929  (0.5, 0.1, 0.2, 0.5)  0.509418  5000
5128  (0.8, 0.1, 0.3, 0.8)  0.509462  5000
4480  (0.7, 0.2, 0.3, 0.8)  0.509693  5000
3821  (0.6, 0.3, 0.2, 0.6)  0.509854  5000
5198  (0.8, 0.2, 0.2, 0.6)  0.509930  5000
2189  (0.4, 0.1, 0.1, 0.3)  0.509930  5000
4471  (0.7, 0.2, 0.2, 0.8)  0.510159  5000
6101  (0.9, 0.4, 0.3, 0.9)  0.510446  5000
4387  (0.7, 0.1, 0.2, 0.5)  0.510519  5000
5849  (0.9, 0.1, 0.2, 0.9)  0.510996  5000
3648  (0.6, 0.1, 0.1, 0.4)  0.51

In [8]:
stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 5.0, 0.5   # ← your true values
percentile = 0.05

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath="../../experimental_data/from_260312/ss_2params_R05p0.csv",
                                   standard_point=(192.95652174, 96.77926067, -0.36994225,  15.68656357),
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0='../../experimental_data/from_260312/R0_samps_2params_R05p0.csv',
                                file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R05p0.csv',
                                true_R0=5.0,
                                true_sigma=0.5, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

chebyshev distance: 
                    weights     score     n
3011  (0.5, 0.2, 0.2, 0.6)  0.477549  5000
3092  (0.5, 0.3, 0.2, 0.6)  0.477549  5000
2930  (0.5, 0.1, 0.2, 0.6)  0.477549  5000
3173  (0.5, 0.4, 0.2, 0.6)  0.480139  5000
5372  (0.8, 0.4, 0.3, 0.9)  0.485904  5000
5129  (0.8, 0.1, 0.3, 0.9)  0.485904  5000
5291  (0.8, 0.3, 0.3, 0.9)  0.485904  5000
5453  (0.8, 0.5, 0.3, 0.9)  0.485904  5000
5210  (0.8, 0.2, 0.3, 0.9)  0.485904  5000
5534  (0.8, 0.6, 0.3, 0.9)  0.486386  5000
5521  (0.8, 0.6, 0.2, 0.5)  0.491903  5000
2688  (0.4, 0.7, 0.2, 0.7)  0.491957  5000
4399  (0.7, 0.1, 0.3, 0.8)  0.492861  5000
4642  (0.7, 0.4, 0.3, 0.8)  0.492861  5000
4480  (0.7, 0.2, 0.3, 0.8)  0.492861  5000
4561  (0.7, 0.3, 0.3, 0.8)  0.492861  5000
4643  (0.7, 0.4, 0.3, 0.9)  0.493479  5000
4400  (0.7, 0.1, 0.3, 0.9)  0.493479  5000
4562  (0.7, 0.3, 0.3, 0.9)  0.493479  5000
4481  (0.7, 0.2, 0.3, 0.9)  0.493479  5000
4238  (0.6, 0.8, 0.3, 0.9)  0.493498  5000
4724  (0.7, 0.5, 0.3, 0.9)  0.49

In [9]:
stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 5.0, 0.5   # ← your true values
percentile = 0.05

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath="../../experimental_data/from_260312/ss_2params_R05p0.csv",
                                   standard_point=(192.95652174, 96.77926067, -0.36994225,  15.68656357),
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0='../../experimental_data/from_260312/R0_samps_2params_R05p0.csv',
                                file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R05p0.csv',
                                true_R0=5.0,
                                true_sigma=0.5, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

minkowski distance: 
                    weights     score     n
6162  (0.9, 0.5, 0.1, 0.7)  0.501451  5000
6324  (0.9, 0.7, 0.1, 0.7)  0.504633  5000
6243  (0.9, 0.6, 0.1, 0.7)  0.505258  5000
6244  (0.9, 0.6, 0.1, 0.8)  0.505748  5000
6406  (0.9, 0.8, 0.1, 0.8)  0.505778  5000
4703  (0.7, 0.5, 0.1, 0.6)  0.507042  5000
6326  (0.9, 0.7, 0.1, 0.9)  0.507282  5000
5514  (0.8, 0.6, 0.1, 0.7)  0.507741  5000
5597  (0.8, 0.7, 0.1, 0.9)  0.507948  5000
6487  (0.9, 0.9, 0.1, 0.8)  0.508465  5000
5432  (0.8, 0.5, 0.1, 0.6)  0.508596  5000
6325  (0.9, 0.7, 0.1, 0.8)  0.509133  5000
6080  (0.9, 0.4, 0.1, 0.6)  0.509146  5000
6081  (0.9, 0.4, 0.1, 0.7)  0.509347  5000
6163  (0.9, 0.5, 0.1, 0.8)  0.509375  5000
5516  (0.8, 0.6, 0.1, 0.9)  0.509702  5000
4868  (0.7, 0.7, 0.1, 0.9)  0.509929  5000
2681  (0.4, 0.7, 0.1, 0.9)  0.510331  5000
5676  (0.8, 0.8, 0.1, 0.7)  0.510516  5000
6000  (0.9, 0.3, 0.1, 0.7)  0.510824  5000
6405  (0.9, 0.8, 0.1, 0.7)  0.510847  5000
4300  (0.6, 0.9, 0.1, 0.8)  0.51

In [10]:
stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 5.0, 0.5   # ← your true values
percentile = 0.05

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath="../../experimental_data/from_260312/ss_2params_R05p0.csv",
                                   standard_point=(192.95652174, 96.77926067, -0.36994225,  15.68656357),
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0='../../experimental_data/from_260312/R0_samps_2params_R05p0.csv',
                                file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R05p0.csv',
                                true_R0=5.0,
                                true_sigma=0.5, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

cosine distance: 
                    weights     score     n
6552  (0.9, 0.9, 0.9, 0.1)  0.637959  5000
3474  (0.5, 0.7, 0.9, 0.1)  0.638196  5000
4590  (0.7, 0.3, 0.7, 0.1)  0.638715  5000
5823  (0.8, 0.9, 0.9, 0.1)  0.639278  5000
6471  (0.9, 0.8, 0.9, 0.1)  0.640849  5000
3780  (0.6, 0.2, 0.7, 0.1)  0.640939  5000
5319  (0.8, 0.3, 0.7, 0.1)  0.641067  5000
6543  (0.9, 0.9, 0.8, 0.1)  0.641080  5000
4356  (0.6, 0.9, 0.8, 0.1)  0.641357  5000
5400  (0.8, 0.4, 0.7, 0.1)  0.641363  5000
4932  (0.7, 0.7, 0.9, 0.1)  0.641388  5000
4671  (0.7, 0.4, 0.7, 0.1)  0.641686  5000
5733  (0.8, 0.8, 0.8, 0.1)  0.641723  5000
3798  (0.6, 0.2, 0.9, 0.1)  0.642012  5000
5814  (0.8, 0.9, 0.8, 0.1)  0.642319  5000
5094  (0.7, 0.9, 0.9, 0.1)  0.642380  5000
3627  (0.5, 0.9, 0.8, 0.1)  0.642384  5000
4365  (0.6, 0.9, 0.9, 0.1)  0.642642  5000
3132  (0.5, 0.3, 0.7, 0.1)  0.642833  5000
3861  (0.6, 0.3, 0.7, 0.1)  0.642936  5000
5742  (0.8, 0.8, 0.9, 0.1)  0.642938  5000
3051  (0.5, 0.2, 0.7, 0.1)  0.64318

In [14]:
##### setting: R0=2.0, sigma=0.5

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 2.0, 0.5   # ← your true values
percentile = 0.05
standard_point=(36.65217391, 18.35906686, -0.34396254,  5.3533094)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R02p0.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R02p0.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R02p0.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
4618  (0.7, 0.4, 0.1, 0.2)  0.105923  5000
6077  (0.9, 0.4, 0.1, 0.3)  0.106069  5000
5996  (0.9, 0.3, 0.1, 0.3)  0.106222  5000
3726  (0.6, 0.2, 0.1, 0.1)  0.106266  5000
5511  (0.8, 0.6, 0.1, 0.4)  0.106660  5000
6078  (0.9, 0.4, 0.1, 0.4)  0.106855  5000
2916  (0.5, 0.1, 0.1, 0.1)  0.106897  5000
6240  (0.9, 0.6, 0.1, 0.4)  0.106961  5000
6085  (0.9, 0.4, 0.2, 0.2)  0.106987  5000
2997  (0.5, 0.2, 0.1, 0.1)  0.107053  5000
5508  (0.8, 0.6, 0.1, 0.1)  0.107097  5000
6158  (0.9, 0.5, 0.1, 0.3)  0.107101  5000
6321  (0.9, 0.7, 0.1, 0.4)  0.107189  5000
6159  (0.9, 0.5, 0.1, 0.4)  0.107302  5000
5592  (0.8, 0.7, 0.1, 0.4)  0.107317  5000
5348  (0.8, 0.4, 0.1, 0.3)  0.107329  5000
5265  (0.8, 0.3, 0.1, 0.1)  0.107898  5000
6004  (0.9, 0.3, 0.2, 0.2)  0.107912  5000
6166  (0.9, 0.5, 0.2, 0.2)  0.108117  5000
4779  (0.7, 0.6, 0.1, 0.1)  0.108123  5000
manhattan distance: 
                    weights     score     n
2840  (0.4

In [15]:
##### setting: R0=2.5, sigma=0.5

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 2.5, 0.5   # ← your true values
percentile = 0.05
standard_point=(74.7826087, 40.32927712, -0.37141329,  9.56328234)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R02p5.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R02p5.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R02p5.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
79    (0.1, 0.1, 0.9, 0.8)  0.208146  5000
80    (0.1, 0.1, 0.9, 0.9)  0.208817  5000
53    (0.1, 0.1, 0.6, 0.9)  0.209442  5000
77    (0.1, 0.1, 0.9, 0.6)  0.209598  5000
805   (0.2, 0.1, 0.9, 0.5)  0.209654  5000
62    (0.1, 0.1, 0.7, 0.9)  0.210129  5000
230   (0.1, 0.3, 0.8, 0.6)  0.210223  5000
240   (0.1, 0.3, 0.9, 0.7)  0.210775  5000
321   (0.1, 0.4, 0.9, 0.7)  0.211248  5000
57    (0.1, 0.1, 0.7, 0.4)  0.211317  5000
795   (0.2, 0.1, 0.8, 0.4)  0.211381  5000
47    (0.1, 0.1, 0.6, 0.3)  0.211476  5000
6480  (0.9, 0.9, 0.1, 0.1)  0.211535  5000
4860  (0.7, 0.7, 0.1, 0.1)  0.211803  5000
797   (0.2, 0.1, 0.8, 0.6)  0.211824  5000
68    (0.1, 0.1, 0.8, 0.6)  0.211959  5000
157   (0.1, 0.2, 0.9, 0.5)  0.212033  5000
67    (0.1, 0.1, 0.8, 0.5)  0.212049  5000
61    (0.1, 0.1, 0.7, 0.8)  0.212145  5000
52    (0.1, 0.1, 0.6, 0.8)  0.212273  5000
58    (0.1, 0.1, 0.7, 0.5)  0.212282  5000
888   (0.2, 0.2, 0.9, 0.7)  0.21

In [16]:
##### setting: R0=3.0, sigma=0.5

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 3.0, 0.5   # ← your true values
percentile = 0.05
standard_point=(102.86956522,  53.44351678,  -0.40986869,  11.23256075)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R03p0.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R03p0.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R03p0.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
1375  (0.2, 0.8, 0.9, 0.8)  0.200767  5000
679   (0.1, 0.9, 0.4, 0.5)  0.201461  5000
156   (0.1, 0.2, 0.9, 0.4)  0.201655  5000
545   (0.1, 0.7, 0.7, 0.6)  0.202226  5000
147   (0.1, 0.2, 0.8, 0.4)  0.202262  5000
454   (0.1, 0.6, 0.6, 0.5)  0.202519  5000
14    (0.1, 0.1, 0.2, 0.6)  0.202568  5000
1130  (0.2, 0.5, 0.9, 0.6)  0.202608  5000
726   (0.1, 0.9, 0.9, 0.7)  0.202783  5000
1859  (0.3, 0.5, 0.9, 0.6)  0.203052  5000
746   (0.2, 0.1, 0.2, 0.9)  0.203061  5000
473   (0.1, 0.6, 0.8, 0.6)  0.203117  5000
554   (0.1, 0.7, 0.8, 0.6)  0.203201  5000
138   (0.1, 0.2, 0.7, 0.4)  0.203366  5000
689   (0.1, 0.9, 0.5, 0.6)  0.203390  5000
1193  (0.2, 0.6, 0.7, 0.6)  0.203412  5000
446   (0.1, 0.6, 0.5, 0.6)  0.203704  5000
635   (0.1, 0.8, 0.8, 0.6)  0.203888  5000
645   (0.1, 0.8, 0.9, 0.7)  0.204276  5000
598   (0.1, 0.8, 0.4, 0.5)  0.204289  5000
463   (0.1, 0.6, 0.7, 0.5)  0.204301  5000
482   (0.1, 0.6, 0.9, 0.6)  0.20

In [17]:
##### setting: R0=3.5, sigma=0.5

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 3.5, 0.5   # ← your true values
percentile = 0.05
standard_point=(126.08695652,  61.16662269,  -0.43314552,  11.4570614)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R03p5.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R03p5.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R03p5.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
3090  (0.5, 0.3, 0.2, 0.4)  0.294864  5000
6099  (0.9, 0.4, 0.3, 0.7)  0.297704  5000
5369  (0.8, 0.4, 0.3, 0.6)  0.298912  5000
4640  (0.7, 0.4, 0.3, 0.6)  0.298921  5000
3009  (0.5, 0.2, 0.2, 0.4)  0.300402  5000
4559  (0.7, 0.3, 0.3, 0.6)  0.300697  5000
5370  (0.8, 0.4, 0.3, 0.7)  0.301124  5000
6018  (0.9, 0.3, 0.3, 0.7)  0.301195  5000
6190  (0.9, 0.5, 0.4, 0.8)  0.301238  5000
1459  (0.3, 0.1, 0.1, 0.2)  0.301868  5000
3738  (0.6, 0.2, 0.2, 0.4)  0.301868  5000
6017  (0.9, 0.3, 0.3, 0.6)  0.301868  5000
6098  (0.9, 0.4, 0.3, 0.6)  0.302188  5000
6109  (0.9, 0.4, 0.4, 0.8)  0.303224  5000
5288  (0.8, 0.3, 0.3, 0.6)  0.303418  5000
4467  (0.7, 0.2, 0.2, 0.4)  0.305453  5000
4721  (0.7, 0.5, 0.3, 0.6)  0.305757  5000
3010  (0.5, 0.2, 0.2, 0.5)  0.306620  5000
3739  (0.6, 0.2, 0.2, 0.5)  0.306658  5000
3820  (0.6, 0.3, 0.2, 0.5)  0.306910  5000
5289  (0.8, 0.3, 0.3, 0.7)  0.306949  5000
4723  (0.7, 0.5, 0.3, 0.8)  0.30

In [18]:
##### setting: R0=4.0, sigma=0.5

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 4.0, 0.5   # ← your true values
percentile = 0.05
standard_point=(154.52173913,  79.70791541,  -0.38680366,  16.05472461)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R04p0.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R04p0.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R04p0.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
1790  (0.3, 0.5, 0.1, 0.9)  1.361537  5000
1142  (0.2, 0.6, 0.1, 0.9)  1.363534  5000
3410  (0.5, 0.7, 0.1, 0.9)  1.363620  5000
1141  (0.2, 0.6, 0.1, 0.8)  1.365756  5000
2681  (0.4, 0.7, 0.1, 0.9)  1.368002  5000
2438  (0.4, 0.4, 0.1, 0.9)  1.368653  5000
8     (0.1, 0.1, 0.1, 0.9)  1.369396  5000
1871  (0.3, 0.6, 0.1, 0.9)  1.370005  5000
882   (0.2, 0.2, 0.9, 0.1)  1.370249  5000
1059  (0.2, 0.5, 0.1, 0.7)  1.370602  5000
1060  (0.2, 0.5, 0.1, 0.8)  1.370840  5000
1061  (0.2, 0.5, 0.1, 0.9)  1.371633  5000
234   (0.1, 0.3, 0.9, 0.1)  1.372837  5000
1223  (0.2, 0.7, 0.1, 0.9)  1.372955  5000
2600  (0.4, 0.6, 0.1, 0.9)  1.374053  5000
3329  (0.5, 0.6, 0.1, 0.9)  1.374473  5000
1708  (0.3, 0.4, 0.1, 0.8)  1.375082  5000
5759  (0.8, 0.9, 0.1, 0.9)  1.375898  5000
2762  (0.4, 0.8, 0.1, 0.9)  1.377794  5000
1709  (0.3, 0.4, 0.1, 0.9)  1.377834  5000
396   (0.1, 0.5, 0.9, 0.1)  1.378127  5000
1304  (0.2, 0.8, 0.1, 0.9)  1.37

In [19]:
##### setting: R0=4.5, sigma=0.5

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 4.5, 0.5   # ← your true values
percentile = 0.05
standard_point=(174.91304348,  87.60703222,  -0.34656069,  16.1122224)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R04p5.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R04p5.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R04p5.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
44    (0.1, 0.1, 0.5, 0.9)  0.367639  5000
35    (0.1, 0.1, 0.4, 0.9)  0.373500  5000
76    (0.1, 0.1, 0.9, 0.5)  0.374932  5000
34    (0.1, 0.1, 0.4, 0.8)  0.375614  5000
25    (0.1, 0.1, 0.3, 0.8)  0.381411  5000
116   (0.1, 0.2, 0.4, 0.9)  0.381782  5000
24    (0.1, 0.1, 0.3, 0.7)  0.382365  5000
764   (0.2, 0.1, 0.4, 0.9)  0.384956  5000
26    (0.1, 0.1, 0.3, 0.9)  0.385048  5000
880   (0.2, 0.2, 0.8, 0.8)  0.391779  5000
30    (0.1, 0.1, 0.4, 0.4)  0.391779  5000
881   (0.2, 0.2, 0.8, 0.9)  0.392042  5000
412   (0.1, 0.6, 0.1, 0.8)  0.396255  5000
800   (0.2, 0.1, 0.8, 0.9)  0.397124  5000
413   (0.1, 0.6, 0.1, 0.9)  0.397412  5000
779   (0.2, 0.1, 0.6, 0.6)  0.399529  5000
970   (0.2, 0.3, 0.9, 0.8)  0.399796  5000
232   (0.1, 0.3, 0.8, 0.8)  0.399956  5000
15    (0.1, 0.1, 0.2, 0.7)  0.400399  5000
233   (0.1, 0.3, 0.8, 0.9)  0.400399  5000
97    (0.1, 0.2, 0.2, 0.8)  0.400554  5000
132   (0.1, 0.2, 0.6, 0.7)  0.40

In [20]:
##### setting: R0=5.0, sigma=0.5

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 5.0, 0.5   # ← your true values
percentile = 0.05
standard_point=(192.95652174,  96.77926067,  -0.36994225,  15.68656357)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R05p0.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R05p0.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R05p0.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
5838  (0.9, 0.1, 0.1, 0.7)  0.490504  5000
5921  (0.9, 0.2, 0.1, 0.9)  0.491479  5000
5840  (0.9, 0.1, 0.1, 0.9)  0.492319  5000
6001  (0.9, 0.3, 0.1, 0.8)  0.492329  5000
5839  (0.9, 0.1, 0.1, 0.8)  0.493461  5000
4380  (0.7, 0.1, 0.1, 0.7)  0.493999  5000
5920  (0.9, 0.2, 0.1, 0.8)  0.494276  5000
4624  (0.7, 0.4, 0.1, 0.8)  0.494457  5000
5191  (0.8, 0.2, 0.1, 0.8)  0.494567  5000
4705  (0.7, 0.5, 0.1, 0.8)  0.494630  5000
5192  (0.8, 0.2, 0.1, 0.9)  0.494765  5000
5353  (0.8, 0.4, 0.1, 0.8)  0.494910  5000
5109  (0.8, 0.1, 0.1, 0.7)  0.495156  5000
4381  (0.7, 0.1, 0.1, 0.8)  0.495347  5000
2600  (0.4, 0.6, 0.1, 0.9)  0.495356  5000
5108  (0.8, 0.1, 0.1, 0.6)  0.495422  5000
6172  (0.9, 0.5, 0.2, 0.8)  0.496040  5000
3001  (0.5, 0.2, 0.1, 0.5)  0.496199  5000
5272  (0.8, 0.3, 0.1, 0.8)  0.496294  5000
4462  (0.7, 0.2, 0.1, 0.8)  0.496469  5000
3002  (0.5, 0.2, 0.1, 0.6)  0.496721  5000
5110  (0.8, 0.1, 0.1, 0.8)  0.49